In [1]:
# ================================================================
# dashboard_scripts.ipynb
# ================================================================
# 목적:
#   project_scripts.ipynb에서 저장된 분석 결과물을 로드하여
#   대시보드 플랫폼(Tableau + Streamlit)에서 사용할
#   데이터 파일을 생성하고 저장합니다.
#
# 실행 전제:
#   project_scripts.ipynb 말미의 저장 코드 실행 완료
#   data/dashboard_source/ 에 아래 파일 존재
#   · master.parquet
#   · tokens_ana.parquet
#   · tokens_ana_comm.parquet
#   · cooccur_item_item.parquet
#   · cooccur_style_style.parquet
#   · cooccur_item_style.parquet
# ================================================================

import pandas as pd
import numpy as np
import shutil
from pathlib import Path

# ── 경로 설정 ─────────────────────────────────────────────────
# dashboard_scripts.ipynb 위치: scripts/
# data 폴더 위치: ../data/

BASE_DIR      = Path("../data")
SOURCE_DIR    = BASE_DIR / "analytics_output"
DASHBOARD_DIR = BASE_DIR / "dashboard"

DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

print("✅ 경로 설정 완료")
print(f"   입력 경로: {SOURCE_DIR.resolve()}")
print(f"   출력 경로: {DASHBOARD_DIR.resolve()}")


# ================================================================
# Cell 1. 데이터 로드
# ================================================================

print("\n⏳ 데이터 로드 중...")

master          = pd.read_parquet(SOURCE_DIR / "master.parquet")
tokens_ana      = pd.read_parquet(SOURCE_DIR / "tokens_ana.parquet")
tokens_ana_comm = pd.read_parquet(SOURCE_DIR / "tokens_ana_comm.parquet")
df_ii           = pd.read_parquet(SOURCE_DIR / "cooccur_item_item.parquet")
df_ss           = pd.read_parquet(SOURCE_DIR / "cooccur_style_style.parquet")
df_is           = pd.read_parquet(SOURCE_DIR / "cooccur_item_style.parquet")

print(f"✅ master          : {len(master):,}행")
print(f"✅ tokens_ana      : {len(tokens_ana):,}행")
print(f"✅ tokens_ana_comm : {len(tokens_ana_comm):,}행")
print(f"✅ cooccur_ii      : {len(df_ii):,}행")
print(f"✅ cooccur_ss      : {len(df_ss):,}행")
print(f"✅ cooccur_is      : {len(df_is):,}행")

# 공통 설정
COMMUNITY_ORDER = ["ZZ", "KK", "FS", "HA", "GO"]
COMMUNITY_MAP   = {
    "ZZ": "Z9DY",
    "KK": "KKST",
    "FS": "FIT THE SIZE",
    "HA": "HAO",
    "GO": "GOCD",
}


# ================================================================
# Cell 2. Tableau 트렌드 분석용 데이터
# ================================================================
# 사용 DataFrame: tokens_ana (2-1 분석 기준)
# 포함 내용:
#   · 연도별 점유율
#   · 시즌별 점유율
#   · 연도×시즌별 점유율
# ================================================================

def get_share(df, group_cols, token_type):
    """
    group_cols 기준으로 token_type 점유율 계산
    """
    sub   = df[df["TOKEN_TYPE"] == token_type].copy()
    total = sub.groupby(group_cols).size().reset_index(name="total")
    cnt   = sub.groupby(group_cols + ["token"]).size().reset_index(name="count")
    cnt   = cnt.merge(total, on=group_cols)
    cnt["share"]      = cnt["count"] / cnt["total"] * 100
    cnt["TOKEN_TYPE"] = token_type
    return cnt[group_cols + ["TOKEN_TYPE", "token", "count", "share"]]


# ── 연도별 점유율 ──────────────────────────────────────────────
df_yearly = pd.concat([
    get_share(tokens_ana, ["YEAR"], t)
    for t in ["BRAND", "ITEM", "STYLE"]
], ignore_index=True)

# ── 시즌별 점유율 ──────────────────────────────────────────────
df_seasonal = pd.concat([
    get_share(tokens_ana, ["SEASON"], t)
    for t in ["BRAND", "ITEM", "STYLE"]
], ignore_index=True)

# ── 연도×시즌별 점유율 (2025 Fall/Winter 제외) ────────────────
tokens_ana_valid = tokens_ana[
    ~((tokens_ana["YEAR"] == 2025) &
      (tokens_ana["SEASON"].isin(["Fall", "Winter"])))
].copy()

df_season_year = pd.concat([
    get_share(tokens_ana_valid, ["SEASON_YEAR", "YEAR", "SEASON"], t)
    for t in ["BRAND", "ITEM", "STYLE"]
], ignore_index=True)

# ── 저장 ──────────────────────────────────────────────────────
df_yearly.to_csv(
    DASHBOARD_DIR / "tableau_trend_yearly.csv",
    index=False, encoding="utf-8-sig"
)
df_seasonal.to_csv(
    DASHBOARD_DIR / "tableau_trend_seasonal.csv",
    index=False, encoding="utf-8-sig"
)
df_season_year.to_csv(
    DASHBOARD_DIR / "tableau_trend_season_year.csv",
    index=False, encoding="utf-8-sig"
)

print("\n✅ Tableau 트렌드 데이터 저장 완료")
print(f"   · tableau_trend_yearly.csv      ({len(df_yearly):,}행)")
print(f"   · tableau_trend_seasonal.csv    ({len(df_seasonal):,}행)")
print(f"   · tableau_trend_season_year.csv ({len(df_season_year):,}행)")


# ================================================================
# Cell 3. Tableau 커뮤니티 세그먼트 분석용 데이터
# ================================================================
# 사용 DataFrame: tokens_ana_comm (2-2 분석 기준)
# 포함 내용:
#   · 커뮤니티별 점유율 + 전체 평균 대비 편차
#   · 커뮤니티 간 Jaccard 유사도
#   · 커뮤니티별 담론 밀도
# ================================================================

# ── 커뮤니티별 점유율 + 편차 ──────────────────────────────────
comm_rows = []
for code in COMMUNITY_ORDER:
    for t in ["BRAND", "ITEM", "STYLE"]:
        sub = tokens_ana_comm[
            (tokens_ana_comm["SOURCE"] == code) &
            (tokens_ana_comm["TOKEN_TYPE"] == t)
        ]
        total = len(sub)
        if total == 0:
            continue
        cnt               = sub.groupby("token").size().reset_index(name="count")
        cnt["share"]      = cnt["count"] / total * 100  # % 단위
        cnt["SOURCE"]     = code
        cnt["TOKEN_TYPE"] = t
        comm_rows.append(cnt)

df_comm = pd.concat(comm_rows, ignore_index=True)

# 전체 평균 점유율 계산 (동일한 스케일: % 단위)
total_share = {}
for t in ["BRAND", "ITEM", "STYLE"]:
    sub = tokens_ana_comm[tokens_ana_comm["TOKEN_TYPE"] == t]
    s   = sub.groupby("token").size()
    total_share[t] = (s / len(sub) * 100).to_dict()  # ← 수정: s.sum() → len(sub)

df_comm["total_share"] = df_comm.apply(
    lambda r: total_share.get(r["TOKEN_TYPE"], {}).get(r["token"], 0), axis=1
)
df_comm["delta"]     = df_comm["share"] - df_comm["total_share"]
df_comm["COMMUNITY"] = df_comm["SOURCE"].map(COMMUNITY_MAP)

# ── 커뮤니티 간 Jaccard 유사도 ────────────────────────────────
brand_pivot = df_comm[df_comm["TOKEN_TYPE"] == "BRAND"].pivot_table(
    index="token", columns="SOURCE", values="share", fill_value=0
)

jaccard_rows = []
for c1 in COMMUNITY_ORDER:
    for c2 in COMMUNITY_ORDER:
        if c1 == c2:
            jaccard_rows.append({
                "source_a": c1, "source_b": c2,
                "comm_a":   COMMUNITY_MAP[c1],
                "comm_b":   COMMUNITY_MAP[c2],
                "jaccard":  1.0
            })
            continue
        top20_a = set(brand_pivot[c1].nlargest(20).index)
        top20_b = set(brand_pivot[c2].nlargest(20).index)
        inter   = len(top20_a & top20_b)
        union   = len(top20_a | top20_b)
        jaccard_rows.append({
            "source_a": c1, "source_b": c2,
            "comm_a":   COMMUNITY_MAP[c1],
            "comm_b":   COMMUNITY_MAP[c2],
            "jaccard":  inter / union if union > 0 else 0
        })

df_jaccard = pd.DataFrame(jaccard_rows)

# ── 커뮤니티별 담론 밀도 ─────────────────────────────────────
density_rows = []
for code in COMMUNITY_ORDER:
    posts = len(master[master["SOURCE"] == code])
    sub   = tokens_ana_comm[tokens_ana_comm["SOURCE"] == code]
    density_rows.append({
        "SOURCE":        code,
        "COMMUNITY":     COMMUNITY_MAP[code],
        "post_count":    posts,
        "brand_density": (sub["TOKEN_TYPE"] == "BRAND").sum() / posts,
        "item_density":  (sub["TOKEN_TYPE"] == "ITEM").sum()  / posts,
        "style_density": (sub["TOKEN_TYPE"] == "STYLE").sum() / posts,
    })

df_density = pd.DataFrame(density_rows)

# ── 저장 ──────────────────────────────────────────────────────
df_comm.to_csv(
    DASHBOARD_DIR / "tableau_comm_share.csv",
    index=False, encoding="utf-8-sig"
)
df_jaccard.to_csv(
    DASHBOARD_DIR / "tableau_comm_jaccard.csv",
    index=False, encoding="utf-8-sig"
)
df_density.to_csv(
    DASHBOARD_DIR / "tableau_comm_density.csv",
    index=False, encoding="utf-8-sig"
)

print("\n✅ Tableau 커뮤니티 데이터 저장 완료")
print(f"   · tableau_comm_share.csv   ({len(df_comm):,}행)")
print(f"   · tableau_comm_jaccard.csv ({len(df_jaccard):,}행)")
print(f"   · tableau_comm_density.csv ({len(df_density):,}행)")


# ================================================================
# Cell 4. Streamlit 키워드 탐색기용 데이터
# ================================================================
# 사용 DataFrame: tokens_ana_comm (더 정제된 버전)
# 포함 내용:
#   · 공출현 데이터 (dashboard_source → dashboard 복사)
#   · 시즌별 키워드 언급량
#   · 커뮤니티별 키워드 언급 비중
# ================================================================

# ── 공출현 데이터 복사 ────────────────────────────────────────
for fname in [
    "cooccur_item_item.parquet",
    "cooccur_style_style.parquet",
    "cooccur_item_style.parquet"
]:
    shutil.copy(SOURCE_DIR / fname, DASHBOARD_DIR / fname)

print("\n✅ 공출현 데이터 복사 완료")
print(f"   · cooccur_item_item.parquet    ({len(df_ii):,}행)")
print(f"   · cooccur_style_style.parquet  ({len(df_ss):,}행)")
print(f"   · cooccur_item_style.parquet   ({len(df_is):,}행)")


# ── 시즌별 키워드 언급량 ─────────────────────────────────────
df_keyword_season = (
    tokens_ana_comm[
        tokens_ana_comm["TOKEN_TYPE"].isin(["BRAND", "ITEM", "STYLE"])
    ]
    .groupby(["TOKEN_TYPE", "token", "YEAR", "SEASON"])
    .size()
    .reset_index(name="count")
)
df_keyword_season["SEASON_YEAR"] = (
    df_keyword_season["YEAR"].astype(str) + "-" +
    df_keyword_season["SEASON"]
)

df_keyword_season.to_parquet(
    DASHBOARD_DIR / "keyword_season_count.parquet", index=False
)
print(f"\n✅ 시즌별 키워드 언급량 저장 완료")
print(f"   · keyword_season_count.parquet ({len(df_keyword_season):,}행)")


# ── 커뮤니티별 키워드 언급 비중 ──────────────────────────────
df_keyword_comm = (
    tokens_ana_comm[
        tokens_ana_comm["TOKEN_TYPE"].isin(["BRAND", "ITEM", "STYLE"])
    ]
    .groupby(["TOKEN_TYPE", "token", "SOURCE"])
    .size()
    .reset_index(name="count")
)

comm_total = (
    tokens_ana_comm[
        tokens_ana_comm["TOKEN_TYPE"].isin(["BRAND", "ITEM", "STYLE"])
    ]
    .groupby(["TOKEN_TYPE", "SOURCE"])
    .size()
    .reset_index(name="total")
)

df_keyword_comm = df_keyword_comm.merge(
    comm_total, on=["TOKEN_TYPE", "SOURCE"]
)
df_keyword_comm["share"]     = (
    df_keyword_comm["count"] / df_keyword_comm["total"] * 100
)
df_keyword_comm["COMMUNITY"] = df_keyword_comm["SOURCE"].map(COMMUNITY_MAP)

df_keyword_comm.to_parquet(
    DASHBOARD_DIR / "keyword_comm_share.parquet", index=False
)
print(f"✅ 커뮤니티별 키워드 언급 비중 저장 완료")
print(f"   · keyword_comm_share.parquet ({len(df_keyword_comm):,}행)")


# ================================================================
# Cell 5. 최종 확인
# ================================================================

print("\n" + "="*55)
print("✅ 전체 대시보드 데이터 준비 완료")
print("="*55)
print(f"\n저장 위치: {DASHBOARD_DIR.resolve()}")
for f in sorted(DASHBOARD_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  · {f.name:<45} {size_mb:.1f} MB")

✅ 경로 설정 완료
   입력 경로: /Users/kimsumin/Desktop/Project/Mens Fashion Data Analysis Project/data/analytics_output
   출력 경로: /Users/kimsumin/Desktop/Project/Mens Fashion Data Analysis Project/data/dashboard

⏳ 데이터 로드 중...
✅ master          : 272,066행
✅ tokens_ana      : 5,836,889행
✅ tokens_ana_comm : 5,836,889행
✅ cooccur_ii      : 1,499행
✅ cooccur_ss      : 1,528행
✅ cooccur_is      : 2,906행

✅ Tableau 트렌드 데이터 저장 완료
   · tableau_trend_yearly.csv      (1,827행)
   · tableau_trend_seasonal.csv    (1,828행)
   · tableau_trend_season_year.csv (7,275행)

✅ Tableau 커뮤니티 데이터 저장 완료
   · tableau_comm_share.csv   (2,245행)
   · tableau_comm_jaccard.csv (25행)
   · tableau_comm_density.csv (5행)

✅ 공출현 데이터 복사 완료
   · cooccur_item_item.parquet    (1,499행)
   · cooccur_style_style.parquet  (1,528행)
   · cooccur_item_style.parquet   (2,906행)

✅ 시즌별 키워드 언급량 저장 완료
   · keyword_season_count.parquet (6,327행)
✅ 커뮤니티별 키워드 언급 비중 저장 완료
   · keyword_comm_share.parquet (2,245행)

✅ 전체 대시보드 데이터 준비 완료

저장 위치: /Users/kimsumi